# Medical Reasoning LLM: Baseline → LoRA Fine-tuning → Evaluation

**Portfolio Project**

This project studies whether lightweight post-training can improve a small language model's performance on medical reasoning tasks.

### Research question

> **Can supervised fine-tuning (SFT) of Qwen3-0.6B on medical reasoning data improve answer quality while preserving a compact, reproducible training pipeline?**

### What this project demonstrates

- Hugging Face dataset loading and preprocessing
- Causal language-model inference
- Baseline evaluation
- Response-only supervised fine-tuning
- LoRA parameter-efficient fine-tuning
- Quantitative evaluation with loss/perplexity and text-similarity metrics
- Error analysis
- Optional LLM-as-a-judge evaluation
- Reproducible experiment tracking

### Model / data

- Base model: `Qwen/Qwen3-0.6B`
- Dataset: `FreedomIntelligence/medical-o1-reasoning-SFT`
- Frameworks: PyTorch, Transformers, PEFT, Datasets

> **Important:** This is a research/portfolio project, not a clinical decision-support system. Model outputs must not be used for patient care.

## 1. Environment Setup

Run this notebook in Google Colab with a GPU runtime (T4 is sufficient for a small LoRA experiment).

The project is intentionally structured so that the expensive training/evaluation cells can be skipped while developing the pipeline.

In [ ]:
# Install dependencies
!pip -q install -U transformers datasets accelerate peft evaluate rouge_score sentencepiece

# Optional: uncomment if you want experiment tracking
# !pip -q install wandb


In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    set_seed,
)

from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Load and Inspect the Medical Reasoning Dataset

The original notebook uses the `FreedomIntelligence/medical-o1-reasoning-SFT` dataset and the fields `Question`, `Complex_CoT`, and `Response`. We retain that setup here. 

In [ ]:
# Load dataset
df = pd.read_json(
    "hf://datasets/FreedomIntelligence/medical-o1-reasoning-SFT/medical_o1_sft.json"
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(3))


In [ ]:
# Basic data-quality checks
required_cols = ["Question", "Complex_CoT", "Response"]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"

print("Missing values:")
display(df[required_cols].isna().sum())

print("\nDuplicate questions:", df["Question"].duplicated().sum())
print("\nExample question:")
print(df.loc[0, "Question"])


In [ ]:
# Create a reproducible train/validation/test split.
# We stratify only by random seed; the dataset is not assumed to have a label column.
df = df[required_cols].dropna().drop_duplicates(subset=["Question"]).reset_index(drop=True)

train_df = df.sample(frac=0.90, random_state=SEED)
remaining = df.drop(train_df.index)
val_df = remaining.sample(frac=0.50, random_state=SEED)
test_df = remaining.drop(val_df.index)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


## 3. Model and Prompting Strategy

The original notebook loads `Qwen/Qwen3-0.6B` and enables thinking mode through the chat template. We keep the same base model and make the prompt format explicit so that baseline and fine-tuned models are evaluated under the same interface.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if not torch.cuda.is_available():
    model = model.to(DEVICE)

model.eval()
print("Loaded:", MODEL_NAME)


In [ ]:
def build_prompt(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )

def generate_answer(model, question, max_new_tokens=512, temperature=0.0):
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
    )

    if temperature and temperature > 0:
        generation_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        generation_kwargs.update(do_sample=False)

    with torch.no_grad():
        generated = model.generate(**generation_kwargs)

    new_tokens = generated[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Smoke test
print(generate_answer(model, test_df.iloc[0]["Question"], max_new_tokens=128))


## 4. Baseline Evaluation

Before training, establish a reproducible baseline.

We measure:

1. **Generation length**
2. **ROUGE-L** against the reference response
3. **Exact normalized match** as a strict, low-sensitivity metric

These are imperfect proxies for medical correctness. Later we add optional LLM-as-a-judge evaluation and qualitative error analysis.

In [ ]:
from evaluate import load

rouge = load("rouge")

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s%./+-]", "", text)
    return text

def evaluate_generations(model, eval_df, max_samples=50, max_new_tokens=256):
    subset = eval_df.head(max_samples).copy()
    rows = []

    for _, row in subset.iterrows():
        pred = generate_answer(
            model,
            row["Question"],
            max_new_tokens=max_new_tokens,
        )
        ref = str(row["Response"])

        rows.append({
            "question": row["Question"],
            "prediction": pred,
            "reference": ref,
            "prediction_length": len(pred.split()),
            "exact_match": int(normalize_text(pred) == normalize_text(ref)),
        })

    results = pd.DataFrame(rows)
    rouge_scores = rouge.compute(
        predictions=results["prediction"].tolist(),
        references=results["reference"].tolist(),
        use_stemmer=True,
    )

    summary = {
        "n": len(results),
        "exact_match": results["exact_match"].mean(),
        "avg_prediction_words": results["prediction_length"].mean(),
        "rougeL": rouge_scores["rougeL"],
    }
    return results, summary

# Start small while developing. Increase to the full test set after the pipeline works.
baseline_results, baseline_summary = evaluate_generations(
    model, test_df, max_samples=20, max_new_tokens=256
)

print(json.dumps(baseline_summary, indent=2))
display(baseline_results.head(5))


## 5. Build the SFT Dataset

The original notebook constructs a target consisting of:

`<think>{reason}</think> + response`

We preserve this supervision signal, but train with **response-target masking** so that the model is optimized only on target tokens rather than accidentally treating the prompt as a prediction target.

For portfolio purposes, this makes the training objective explicit and reproducible.

In [ ]:
MAX_LENGTH = 1024

def make_target(row):
    return f"<think>{row['Complex_CoT']}</think>\n{row['Response']}"

def tokenize_example(row):
    prompt = build_prompt(row["Question"])
    target = make_target(row)

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]

    # Reserve space for the target and truncate the prompt if necessary.
    if len(prompt_ids) + len(target_ids) > MAX_LENGTH:
        prompt_ids = prompt_ids[: max(0, MAX_LENGTH - len(target_ids))]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))

train_tok = train_ds.map(
    tokenize_example,
    remove_columns=train_ds.column_names,
)
val_tok = val_ds.map(
    tokenize_example,
    remove_columns=val_ds.column_names,
)

print(train_tok)
print(val_tok)


## 6. LoRA Fine-tuning

Instead of full-parameter fine-tuning, use **LoRA (Low-Rank Adaptation)**.

Why LoRA?

- Lower memory requirement
- Faster experimentation
- Preserves the original base model
- Makes ablation and reproducibility easier
- Appropriate for a portfolio-scale experiment on a Colab GPU

The exact target modules can vary by model architecture; the cell below inspects Qwen's linear modules and uses the attention projections that are standard for this architecture.

In [ ]:
# Reload a clean copy for fine-tuning
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if not torch.cuda.is_available():
    base_model = base_model.to(DEVICE)

base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

OUTPUT_DIR = "./medical_reasoning_qwen3_lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

# Uncomment to train.
# train_output = trainer.train()
# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)


## 7. Post-training Evaluation

After training, reload the LoRA adapter and evaluate the **same test set** used for the baseline.

The key comparison is:

> **Base Qwen3-0.6B vs LoRA fine-tuned Qwen3-0.6B**

Do not tune hyperparameters on the test set.

In [ ]:
# If training was completed, evaluate the trained adapter.
# Otherwise, keep this cell commented while developing the pipeline.

# lora_model.eval()
# tuned_results, tuned_summary = evaluate_generations(
#     lora_model, test_df, max_samples=20, max_new_tokens=256
# )
#
# comparison = pd.DataFrame([baseline_summary, tuned_summary], index=["Base", "LoRA"])
# display(comparison)


## 8. Optional LLM-as-a-Judge Evaluation

Text similarity does not equal medical correctness. A generated answer can be semantically correct while using different wording from the reference.

For a stronger portfolio project, add a **separate judge evaluation** that scores answer correctness against the reference answer.

Recommended scoring rubric:

- **0 = incorrect**
- **1 = partially correct**
- **2 = substantially correct**
- **3 = fully correct**

Keep the judge prompt fixed and report the model, rubric, sample size, and judge limitations.

Do not send protected health information or real patient data to an external model.

In [ ]:
# Optional: OpenAI judge scaffold.
# Set OPENAI_API_KEY in the environment before running.
#
# !pip -q install openai
# from openai import OpenAI
#
# client = OpenAI()
#
# JUDGE_PROMPT = '''
# You are evaluating a medical QA model for research purposes.
# Compare the predicted answer with the reference answer.
# Score correctness from 0 to 3:
# 0 = incorrect
# 1 = partially correct / major omission
# 2 = mostly correct / minor issue
# 3 = fully correct
# Return ONLY the integer score.
# '''
#
# def judge_answer(prediction, reference):
#     response = client.responses.create(
#         model="gpt-5.6",
#         input=[
#             {"role": "system", "content": JUDGE_PROMPT},
#             {"role": "user", "content": f"Reference:\n{reference}\n\nPrediction:\n{prediction}"}
#         ],
#     )
#     return int(response.output_text.strip())


## 9. Error Analysis

A strong ML portfolio should explain **where the model fails**, not only report one aggregate score.

Suggested error categories:

1. Factual error
2. Incomplete answer
3. Reasoning error
4. Hallucination
5. Instruction-following error
6. Excessive verbosity
7. Correct conclusion with weak/incorrect reasoning

Review a fixed sample of errors and summarize the dominant failure modes.

In [ ]:
# Example error-analysis table
# Replace with manually reviewed examples after running baseline and LoRA models.

error_analysis = pd.DataFrame([
    # {
    #     "question": "...",
    #     "prediction": "...",
    #     "reference": "...",
    #     "category": "factual error",
    #     "notes": "..."
    # }
])

if len(error_analysis):
    display(error_analysis)
else:
    print("Add manually reviewed examples here after evaluation.")


## 10. Ablation Plan

To make the project research-quality, run a small set of controlled experiments.

| Experiment | Change | Purpose |
|---|---|---|
| E0 | Base Qwen3-0.6B | Baseline |
| E1 | LoRA SFT | Measure post-training benefit |
| E2 | Lower LoRA rank | Parameter-efficiency ablation |
| E3 | No reasoning target | Test whether reasoning supervision helps |
| E4 | Different max sequence length | Test context/truncation effects |

The primary comparison should be made on the **same held-out test set**.

## 11. Results Table

Populate this table after experiments.

Recommended metrics:

- ROUGE-L
- exact match
- judge score (0–3)
- validation loss
- perplexity
- average generation length

A good result section should report both improvements and regressions.

In [ ]:
results_table = pd.DataFrame({
    "Model": ["Qwen3-0.6B Base", "Qwen3-0.6B + LoRA"],
    "ROUGE-L": [np.nan, np.nan],
    "Exact Match": [np.nan, np.nan],
    "Judge Score (0-3)": [np.nan, np.nan],
    "Val Loss": [np.nan, np.nan],
    "Perplexity": [np.nan, np.nan],
})

display(results_table)


## 12. Reproducibility Checklist

Before publishing:

- [ ] Record Python / Transformers / PEFT versions
- [ ] Record GPU type
- [ ] Fix random seeds
- [ ] Keep train/validation/test split fixed
- [ ] Keep the test set untouched during tuning
- [ ] Save LoRA adapter weights
- [ ] Save tokenizer
- [ ] Export evaluation results as CSV
- [ ] Include the final configuration in the README
- [ ] Document known limitations


In [ ]:
import sys
import transformers
import datasets
import peft

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)


## 13. Portfolio Summary

### Suggested GitHub title

**Medical Reasoning LLM Post-training with Qwen3-0.6B**

### One-line summary

> Built a reproducible medical reasoning LLM pipeline comparing a Qwen3-0.6B baseline with LoRA-based supervised fine-tuning, using held-out evaluation, text-similarity metrics, optional LLM-based correctness scoring, and qualitative error analysis.

### Resume bullet template

> **Medical Reasoning LLM Post-training | PyTorch, Hugging Face, PEFT, Qwen3** — Built an end-to-end evaluation and LoRA fine-tuning pipeline for a 0.6B medical reasoning LLM; benchmarked baseline vs. post-trained performance on a held-out test set using ROUGE-L, exact match, and structured correctness evaluation, with error analysis and controlled ablations.

Replace the final bullet with **your actual measured results** after the experiments are completed. Do not claim an improvement until it has been measured.